### 1. Install Twikit

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2. Login

In [ ]:
!pip install twikit
from twikit import Client, TwitterException

async def init_client(auth_info_1, auth_info_2, password):
    client = Client('en-US')
    cookies_path = 'cookies.json'

    try:
        client.load_cookies(cookies_path)
        print("Cookies ditemukan, tidak perlu login ulang.")
    except FileNotFoundError:
        print("Cookies tidak ditemukan, login ulang.")

        await client.login(
            auth_info_1=auth_info_1,
            auth_info_2=auth_info_2,
            password=password,
            cookies_file=cookies_path
        )

        client.save_cookies(cookies_path)
        print("Login berhasil dan cookies disimpan.")
    except TwitterException as e:
        print(e.msg)
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    return client


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.8/611.8 kB 47.1 MB/s eta 0:00:00
  Created wheel for pyjsparser: filename=pyjsparser-2.7.1-py3-none-any.whl size=25982 sha256=6a958986032dd33b069f1080cff121f3ae8b8dcc3bcb432c8c784596eb0a8acf
  Stored in directory: /root/.cache/pip/wheels/14/32/1d/9ef7b582e358446aeef4b9052aa89ef4dffa1688c1aae8aa13
Successfully built pyjsparser


In [ ]:
import asyncio
import nest_asyncio

try:
    nest_asyncio.apply()
except RuntimeError:
    pass

client = asyncio.get_event_loop().run_until_complete(
    init_client(
        auth_info_1='@pengenluluscpt6', #Username
        auth_info_2='evoscecen@gmail.com', #Mail/Phone/etc
        password='Evos1234',
    )
)

Cookies tidak ditemukan, login ulang.
Login berhasil dan cookies disimpan.


### 3. Scrape

In [ ]:
import pandas as pd
import os

nest_asyncio.apply()

CSV_PATH = '/content/drive/MyDrive/Skripsi/Dataset/FIX DATASET/Data20uji.csv'
CURSOR_PATH = 'last_cursor.txt'

def save_batch_to_csv(batch, csv_path):
    df_batch = pd.DataFrame(batch)
    write_header = not os.path.exists(csv_path)
    df_batch.to_csv(csv_path, mode='a', header=write_header, index=False, encoding='utf-8-sig')

def save_cursor(cursor, cursor_path):
    with open(cursor_path, 'w') as f:
        f.write(cursor if cursor else '')

def load_cursor(cursor_path):
    if os.path.exists(cursor_path):
        with open(cursor_path, 'r') as f:
            cursor = f.read().strip()
            return cursor if cursor else None
    return None


In [ ]:
import asyncio
import random
from twikit import AccountSuspended

async def scrape_tweets(client, query, product='Latest'):
    cursor = load_cursor(CURSOR_PATH)
    print(f"Mulai scraping. Cursor terakhir: {cursor}")

    while True:
        try:
            result = await client.search_tweet(query, product, count=20, cursor=cursor)
            batch = list(result)
            if not batch:
                print("Tidak ada tweet lagi.")
                break

            data = []
            for tweet in batch:
                data.append({
                    'username': tweet.user.name,
                    'screen_name': tweet.user.screen_name,
                    'text': tweet.text,
                    'created_at': tweet.created_at,
                    'retweets': tweet.retweet_count,
                    'likes': tweet.favorite_count
                })

            save_batch_to_csv(data, CSV_PATH)
            print(f"Batch {len(data)} tweet disimpan ke CSV.")

            cursor = getattr(result, "next_cursor", None)
            save_cursor(cursor, CURSOR_PATH)

            if not cursor:
                print("Tidak ada cursor untuk batch berikutnya, scraping selesai.")
                break

            delay = random.uniform(15, 30)
            await asyncio.sleep(delay)
        except AccountSuspended as e:
            print("Rate limit. Tidur 60 detik...")
            await asyncio.sleep(60)
            continue
        except Exception as e:
            print(f"Terjadi kesalahan: {e}")
            break

In [ ]:
query = '"danantara" OR "#danantara" until:2025-05-30 since:2025-05-01 lang:id'

await scrape_tweets(client, query)
print("Scraping selesai/manual stop. Data sudah tersimpan bertahap di CSV.")


Mulai scraping. Cursor terakhir: None
Batch 19 tweet disimpan ke CSV.
Batch 20 tweet disimpan ke CSV.
Batch 20 tweet disimpan ke CSV.
Batch 19 tweet disimpan ke CSV.
Batch 15 tweet disimpan ke CSV.
Batch 14 tweet disimpan ke CSV.
Batch 16 tweet disimpan ke CSV.
Batch 16 tweet disimpan ke CSV.
Batch 6 tweet disimpan ke CSV.
Batch 7 tweet disimpan ke CSV.


CancelledError: 